# MLP for sequence learning

Here, I implemented a small neural network in PyTorch for sequence-level prediction.

Steps:
```text
                RNA sequence
                   "AUGC"
                     │
                     ▼
              One-hot encoding
            (seq_len × 4 matrix)

        A → [1 0 0 0]
        U → [0 1 0 0]
        G → [0 0 1 0]
        C → [0 0 0 1]

                     │
                     ▼
           ┌───────────────────┐
           │   Linear layer    │
           │      fc1          │
           │     4 → 16        │
           └───────────────────┘
                     │
                     ▼
                 ReLU
                     │
                     ▼
        Hidden representation per nucleotide
                shape: (seq_len, 16)

           [h11 h12 ... h1,16]
           [h21 h22 ... h2,16]
           [h31 h32 ... h3,16]
           [h41 h42 ... h4,16]

                     │
                     ▼
            Mean pooling over sequence
               average across rows

                shape: (16)

           [mean(h1), mean(h2), ... mean(h16)]

                     │
                     ▼
           ┌───────────────────┐
           │   Linear layer    │
           │      fc2          │
           │     16 → 1        │
           └───────────────────┘
                     │
                     ▼
              Scalar prediction

               expression score
                   e.g. 0.73
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as n

In [2]:
def one_hot_encode(inseq):
    # LETTERS = "AUCG" # letters are hard-coded
    onehot = {"A": [1,0,0,0],
         "U": [0,1,0,0],
         "C": [0,0,1,0],
         "G": [0,0,0,1]}
    return torch.tensor([onehot[s] for s in inseq], dtype=torch.float32)


In [3]:
class SeqModel(nn.Module):
    def __init__(self):
        # it runs first nn.Module.__init__()
        # and then it rruns SeqModel.__init__()
        super().__init__() # calls the constructor of the parent class (nn.Module); 
                           # starts the engine of the neural network framework
                           # Run the initialization logic of the parent class.
                           # otehrwise model.parameters() wont do anything
        # Each of the 16 outputs is a weighted combination of A,U,G,C
        # Example neuron: feature1 = 0.3*A + 0.1*U + 0.8*G - 0.2*C
        self.fc1 = nn.Linear(4,16) 
        self.fc2 = nn.Linear(16,1)
    def forward(self, x):
        x = self.fc1(x) # 16 learned features per position
        x = torch.relu(x) # this introduces non-linearity.
        x = x.mean(dim=0) # We average across the sequence length over that 16 features (that is per position in a sequence)
        x = self.fc2(x) # scalar prediction
        return x

In [14]:
SEED = 42
torch.manual_seed(SEED) # for reproducibility

# Let's test the model with a random input:
model = SeqModel()
x = one_hot_encode("AAAUGCC")
prediction = model(x)
print(f"Prediction: {prediction.item():.2f}")

# We got the prediction, but it is not good.
# We need to train the model to make better predictions.
target = torch.tensor([0.8])
# Loss function
loss_fn = nn.MSELoss() # mean squared error loss, i.e. (prediction − target)²
loss = loss_fn(prediction, target)
print(f"Loss: {loss.item():.2f}") # Let's check how wrong the prediction is;
# 0.52 just means the prediction is far from the target.
# that is normal for untrained network. We will train the network to make better predictions.

Prediction: 0.08
Loss: 0.52


In [ ]:
# What could help us is backpropagation:
"""
sequence
↓
one-hot encoding
↓
model
↓
prediction
↓
compare to target
↓
loss
↓
update weights
"""

# We use backpropagation to update weights int he model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.zero_grad() # clear previous gradients; PyTorch accumulates gradients, so we reset them each iteration.
prediction = model(x) # forward pass to get the prediction
loss = loss_fn(prediction, target) # Measure how wrong the prediction is.
loss.backward() # Compute gradients for all weights.
optimizer.step() #  Optimizer updates the weights using those gradients.

loss_fn = nn.MSELoss()
with torch.no_grad():
    pred = model(x)
    loss = loss_fn(pred, target)
print(f"Loss: {loss.item():.2f}, Prediction: {pred.item():.2f}, Target: {target.item():.2f}")
# So the model learned a little, but only a little (0.08 -> Prediction: 0.12, target: 0.80).
# What we cna do is to train for many epochs, and not only one step!

Loss: 0.47, Prediction: 0.12, target: 0.80


In [ ]:
#  -------- toy example with multiple sequences and targets
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [
    0.8,
    0.3,
    0.6
]

# -------- prepare data --------
X = [one_hot_encode(seq) for seq in sequences]
Y = torch.tensor(targets, dtype=torch.float32)

# -------- model / loss / optimizer --------
model = SeqModel()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------- training loop --------
for epoch in range(30): # one pass through the entire dataset
    total_loss = 0.0
    for x, y in zip(X,Y):
        optimizer.zero_grad()
        prediction = model(x) # forward pass to get the prediction
        loss = loss_fn(prediction, y) # Measure how wrong the prediction is.
        loss.backward() # Compute gradients for all weights.
        optimizer.step() #  Optimizer updates the weights using those gradients.
        total_loss += loss.item()
    avg_loss = total_loss / len(X)
    print(f"epoch={epoch}, avg_loss={avg_loss:.2f}")

epoch=0, avg_loss=0.78
epoch=1, avg_loss=0.75
epoch=2, avg_loss=0.73
epoch=3, avg_loss=0.71
epoch=4, avg_loss=0.69
epoch=5, avg_loss=0.66
epoch=6, avg_loss=0.64
epoch=7, avg_loss=0.62
epoch=8, avg_loss=0.60
epoch=9, avg_loss=0.58
epoch=10, avg_loss=0.56
epoch=11, avg_loss=0.55
epoch=12, avg_loss=0.53
epoch=13, avg_loss=0.51
epoch=14, avg_loss=0.49
epoch=15, avg_loss=0.48
epoch=16, avg_loss=0.46
epoch=17, avg_loss=0.44
epoch=18, avg_loss=0.43
epoch=19, avg_loss=0.41
epoch=20, avg_loss=0.40
epoch=21, avg_loss=0.39
epoch=22, avg_loss=0.37
epoch=23, avg_loss=0.36
epoch=24, avg_loss=0.34
epoch=25, avg_loss=0.33
epoch=26, avg_loss=0.32
epoch=27, avg_loss=0.31
epoch=28, avg_loss=0.30
epoch=29, avg_loss=0.28


If we shuffle the nucleotides in the sequence, will the model's prediction change? In our current architecture, the prediction will not change much (and theoretically can become identical). Mean pooling removes sequence order.

The model cannot learn motifs like: AUG, TATA, CpG. How real sequence models fix this? They use other architectures like CNNs or transformers!